In [1]:
from util import import_ragas_custom, load_env_variables_from_all_env_files

import_ragas_custom('ragas_custom_2')

Arquivos copiados com sucesso!


In [2]:
load_env_variables_from_all_env_files()

In [3]:
import os
import asyncio
import nest_asyncio

from llama_index.core import SimpleDirectoryReader
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import default_query_distribution
from llama_index.llms.openai import OpenAI
from ragas.run_config import RunConfig
from ragas.llms import LlamaIndexLLMWrapper

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jmess\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
c:\Users\jmess\miniconda3\envs\rag_test\Lib\site-packages\ragas\metrics\base.py:496: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):


In [4]:
nest_asyncio.apply()

In [5]:
DATA = 'data'
LANGUAGE = "portuguese"
TIMEOUT = 2400.0
CACHE_DIR = 'cache_2'
AMOUNT_TESTS = 128
MODEL = 'gpt-4o-mini-2024-07-18'

In [6]:
run_config = RunConfig(max_workers=1)

In [7]:
llm = LlamaIndexLLMWrapper(OpenAI(model='MODEL'), run_config=run_config)

generator = TestsetGenerator(llm)

In [8]:
query_distribution = default_query_distribution(llm)

In [10]:
for query, _ in query_distribution:
    path = os.path.join(CACHE_DIR, query.__class__.__name__)
    if not os.path.exists(path):
        os.makedirs(path)

    try:
        prompts = query.load_prompts(path, LANGUAGE)
        query.set_prompts(**prompts)
    except Exception:
        prompts = asyncio.run(query.adapt_prompts(LANGUAGE, None, True, True))
        query.set_prompts(**prompts)
        query.save_prompts(path)
        prompts = query.load_prompts(path, LANGUAGE)
        query.set_prompts(**prompts)


UnboundLocalError: cannot access local variable 'translated_strings' where it is not associated with a value

In [10]:
documents = SimpleDirectoryReader(DATA).load_data()

In [ ]:
testset = generator.generate_with_llamaindex_docs(documents, AMOUNT_TESTS, query_distribution=query_distribution, with_debugging_logs=True, run_config=run_config)

Applying [SummaryExtractor, HeadlinesExtractor]:   0%|          | 0/24 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/12 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, KeyphrasesExtractor, TitleExtractor]:   0%|          | 0/129 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryCosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating common_concepts:   0%|          | 0/1 [00:00<?, ?it/s]

Generating common themes:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
testset.to_jsonl('testset_openai_4omini.jsonl')